In [ ]:
# ================================================================
# FINAL — Guaranteed, Context-Aware Hallucination Injector (Colab-ready)
# Output columns fixed: filename, hallucination_type, ast_nodes, original_code, hallucinated_code
# ================================================================
# Run prerequisites in Colab:
# !pip install -q tree_sitter==0.21.3 tree_sitter_languages pandas tqdm

import re, random, hashlib
import pandas as pd
from tree_sitter import Parser
from tree_sitter_languages import get_language

# ---------------- CONFIG ----------------
CSV_PATH = "/content/all_cpp_codes_dataset.csv"
OUTPUT_CSV = "/content/final_context_inferred_hallucinated_dataset.csv"
LOG_CSV = "/content/final_context_inferred_log.csv"
RND_SEED = 42
random.seed(RND_SEED)

# ---------------- parser ----------------
CPP = get_language("cpp")
parser = Parser()
parser.set_language(CPP)

# ---------------- utils ----------------
def sha1(s: str) -> str:
    return hashlib.sha1(s.encode('utf-8')).hexdigest()

def remove_comments_and_headers(code: str) -> str:
    c = re.sub(r'/\*[\s\S]*?\*/|//.*', '', code)
    lines=[]
    for L in c.splitlines():
        s=L.strip()
        if s.startswith('#'): continue
        if s.startswith('using ') or s.startswith('namespace '): continue
        lines.append(L)
    return "\n".join(lines)

def safe_text(node, src):
    try:
        return src[node.start_byte:node.end_byte].decode('utf8',errors='ignore')
    except:
        return ''

# =========================================================
# ✅ AST EXTRACTION
# =========================================================
def get_ast_nodes(code):
    """Extract major AST node types from C++ source."""
    try:
        tree = parser.parse(bytes(code, "utf8"))
        root = tree.root_node
        out = []
        def walk(node):
            if node.type in (
                "function_definition",
                "if_statement",
                "for_statement",
                "while_statement",
                "return_statement"
            ):
                out.append(node.type)
            for c in node.children:
                walk(c)
        walk(root)
        return out
    except Exception:
        return []

# =========================================================
# 🧩 Helper utilities (unchanged)
# =========================================================
def extract_identifiers(code: str, topk=30):
    ids = re.findall(r'\b([A-Za-z_][A-Za-z0-9_]*)\b', code)
    CPP_KEYWORDS = set(["int","float","double","long","short","char","bool","void","auto","return","for","while","if","else","switch","case","break","continue","using","namespace","std","cout","cin","include","main","size_t","const","static","struct","class","template"])
    filtered=[i for i in ids if i not in CPP_KEYWORDS]
    seen=[]
    for i in filtered:
        if i not in seen: seen.append(i)
    return seen[:topk]

def find_first_function_body_index(lines):
    joined = "\n".join(lines)
    m = re.search(r'\b[A-Za-z_][A-Za-z0-9_<>:,\s\*]*\b\s+[A-Za-z_][A-Za-z0-9_]*\s*\([^\)]*\)\s*\{', joined)
    if not m:
        return -1
    prefix = joined[:m.end()]
    return prefix.count("\n")

def find_main_index(lines):
    for i,L in enumerate(lines):
        if re.search(r'\bint\s+main\s*\(', L):
            return i
    return -1

# =========================================================
# DEAD injection (unchanged)
# =========================================================
def build_dead_snippet(kind, base_ident):
    if kind=="vector_helper":
        return ("// [HALLUCINATION: injection start - vector_helper]\n"
                f"static std::vector<int> {base_ident}_helper(const std::vector<int>& v) {{ std::vector<int> out(v.size()); for(size_t i=0;i<v.size();++i) out[i]=v[i]; return out; }}\n"
                "// [HALLUCINATION: injection end]\n")
    if kind=="math_helper":
        return ("// [HALLUCINATION: injection start - math_helper]\n"
                f"static int {base_ident}_factor(int n) {{ if(n<=1) return 1; int r=1; for(int i=2;i<=n;++i) if(n%i==0) r*=i; return r; }}\n"
                "// [HALLUCINATION: injection end]\n")
    if kind=="iffalse":
        return ("// [HALLUCINATION: injection start - iffalse]\n"
                "if(false) {\n"
                "    int _tmp_sum = 0; for(int _i=0; _i<8; ++_i) _tmp_sum += _i;\n"
                "}\n"
                "// [HALLUCINATION: injection end]\n")
    return ("// [HALLUCINATION: injection start - unused_const_class]\n"
            "static const int _HALL_UNUSED_CONST = 12345;\n"
            "class _HallucUnused { public: int compute(int x){ return x+1; } }; \n"
            "// [HALLUCINATION: injection end]\n")

def select_dead_kind(cleaned, ids):
    low = cleaned.lower()
    if any(k in low for k in ["vector","array","matrix","grid","vec","cols","rows","push_back"]):
        return "vector_helper"
    if any(k in low for k in ["prime","gcd","factor","lcm"]):
        return "math_helper"
    if any(k.endswith("s") or k.endswith("vec") or k.endswith("arr") for k in ids[:6]):
        return "vector_helper"
    return "iffalse"

def inject_dead_contextual(code: str, ids=None, ast_root=None):
    ids = ids or extract_identifiers(code)
    cleaned = remove_comments_and_headers(code)
    kind = select_dead_kind(cleaned, ids)
    base_ident = ids[0] if ids else "aux"
    snippet = build_dead_snippet(kind, base_ident)
    lines = code.splitlines()
    fn_idx = find_first_function_body_index(lines)
    if fn_idx >= 0:
        insert_at = min(fn_idx+1, len(lines))
        lines.insert(insert_at, snippet)
        return "\n".join(lines)
    main_idx = find_main_index(lines)
    if main_idx >= 0:
        lines.insert(main_idx, snippet)
        return "\n".join(lines)
    first_non_header = 0
    for i,L in enumerate(lines):
        if not L.strip().startswith('#') and not L.strip().startswith('using') and not L.strip().startswith('namespace'):
            first_non_header = i
            break
    insert_at = max(first_non_header, len(lines))
    lines.insert(insert_at, snippet)
    return "\n".join(lines)

# =========================================================
# INCONSISTENCY injection (unchanged)
# =========================================================
def inject_inconsistency_strict(code: str):
    lines = code.splitlines()
    for i,L in enumerate(lines):
        s=L.strip()
        if not s or s.startswith("#") or "template" in s or "using " in s or "namespace " in s: continue
        m = re.search(r'\b(if|for|while)\s*\((.*?)\)', L)
        if not m: continue
        cond = m.group(2)
        if '==' in cond:
            new = cond.replace('==','!= /*HALL:flipped*/',1)
        elif '!=' in cond:
            new = cond.replace('!=','== /*HALL:flipped*/',1)
        elif '<=' in cond:
            new = cond.replace('<=','> /*HALL:flipped*/',1)
        elif '>=' in cond:
            new = cond.replace('>=','< /*HALL:flipped*/',1)
        elif ('<' in cond) and ('<<' not in cond):
            new = re.sub(r'(?<!<)<(?!<)','> /*HALL:flipped*/',cond, count=1)
        elif ('>' in cond) and ('>>' not in cond):
            new = re.sub(r'(?<!>)>(?!>)','< /*HALL:flipped*/',cond, count=1)
        else:
            continue
        lines[i] = L.replace("("+cond+")","("+new+")",1)
        return "\n".join(lines)
    return "\n".join(lines)

# =========================================================
# REPETITION injection (unchanged)
# =========================================================
def inject_repetition_contextual(code: str):
    patterns = [r'(for\s*\([^\)]*\)\s*\{[\s\S]*?\})',
                r'(while\s*\([^\)]*\)\s*\{[\s\S]*?\})',
                r'(if\s*\([^\)]*\)\s*\{[\s\S]*?\})']
    candidates=[]
    for pat in patterns:
        for m in re.finditer(pat, code):
            block = m.group(1)
            candidates.append((len(block), block))
    if not candidates:
        return code
    chosen = max(candidates, key=lambda x: x[0])[1]
    wrapped = (
        "// [HALLUCINATION: repetition start]\n"
        + chosen +
        "\n// [HALLUCINATION: repetition end]\n"
    )
    return code.replace(chosen, chosen + "\n" + wrapped, 1)

# =========================================================
# OMISSION injection (unchanged)
# =========================================================
def inject_omission(code: str, identifiers_list=None, context_keywords=None) -> str:
    lines = code.splitlines()
    candidate_idxs = []
    for i, L in enumerate(lines):
        if L.strip().startswith('#'):
            continue
        if '||' in L or '&&' in L or re.search(r'==\s*0|==\s*NULL|==\s*nullptr|<=\s*0', L):
            candidate_idxs.append(i)
    if not candidate_idxs:
        return code
    idx = candidate_idxs[0]
    L = lines[idx]
    if '||' in L:
        parts = L.split('||', 1)
        lines[idx] = parts[0] + '/* HALL: omitted right operand */'
    elif '&&' in L:
        parts = L.split('&&', 1)
        lines[idx] = parts[0] + '/* HALL: omitted right operand */'
    else:
        removed = lines.pop(idx)
        lines.append('// [HALLUCINATION: omitted line] ' + removed.strip())
    return "\n".join(lines)

# =========================================================
# Scoring + Equal Split Logic
# =========================================================
def score_suitability(code: str):
    cleaned = remove_comments_and_headers(code)
    ast_nodes = get_ast_nodes(cleaned)
    ids = extract_identifiers(cleaned, topk=15)
    has_compound = '||' in cleaned or '&&' in cleaned
    summary = {
        'functions': ast_nodes.count('function_definition'),
        'loops': ast_nodes.count('for_statement') + ast_nodes.count('while_statement'),
        'conditionals': ast_nodes.count('if_statement'),
        'returns': ast_nodes.count('return_statement')
    }
    scores={}
    scores['dead'] = 2*summary['functions'] + (1 if ids else 0)
    scores['inc']  = 3*summary['conditionals'] + (3 if has_compound else 0)
    scores['rep']  = 3*summary['loops'] + (2 if summary['conditionals'] else 0)
    scores['omit'] = (4 if has_compound else 0)
    for k in scores: scores[k] += random.random()*0.1
    meta = {'ast_nodes': ast_nodes, 'identifiers': ids, 'cleaned': cleaned}
    return scores, meta

def balanced_select_indices(df_codes, n_per_group):
    n=len(df_codes)
    all_scores=[]; metas=[]
    for c in df_codes:
        s,m = score_suitability(c)
        all_scores.append(s); metas.append(m)
    labels=['dead','inc','rep','omit']
    assigned=set(); selection={lab:[] for lab in labels}
    for lab in labels:
        scored_list=[(all_scores[i][lab], i) for i in range(len(df_codes))]
        scored_list.sort(reverse=True, key=lambda x:(x[0], -x[1]))
        for score, idx in scored_list:
            if len(selection[lab])>=n_per_group: break
            if idx in assigned: continue
            selection[lab].append(idx); assigned.add(idx)
    return selection, metas

# =========================================================
# MAIN (output columns fixed)
# =========================================================
def main():
    df = pd.read_csv(CSV_PATH)
    if 'code' not in df.columns:
        df['code'] = df.iloc[:,0].astype(str)
    codes = df['code'].astype(str).tolist()
    filenames = df['filename'].astype(str).tolist() if 'filename' in df.columns else [f'code_{i+1:03d}.cpp' for i in range(len(codes))]
    n = len(codes)
    if n < 1:
        raise ValueError("No codes found.")

    # ✅ Equal 5-way split (92–93 each for 463)
    base = n // 5
    rem = n % 5
    labels = ['dead', 'inc', 'rep', 'omit', 'clean']
    target_counts = {l: base + (1 if i < rem else 0) for i, l in enumerate(labels)}

    selection_map, metas = balanced_select_indices(codes, target_counts['dead'])
    assigned = set(idx for lst in selection_map.values() for idx in lst)
    remaining = [i for i in range(n) if i not in assigned]
    random.shuffle(remaining)
    clean_selected = set(remaining[:target_counts['clean']])

    index_to_label = {}
    for lab, lst in selection_map.items():
        for idx in lst:
            index_to_label[idx] = lab
    for idx in clean_selected:
        index_to_label[idx] = 'clean'

    rows=[]; logs=[]
    counts={l:0 for l in labels}

    for i in range(n):
        orig = codes[i]; fname = filenames[i]
        assigned_label = index_to_label.get(i,'clean')
        meta = metas[i] if i < len(metas) else {}
        ids = meta.get('identifiers', extract_identifiers(orig))
        ast_nodes = meta.get('ast_nodes', [])
        mutated = orig
        try:
            if assigned_label == 'dead':
                mutated = inject_dead_contextual(orig, ids)
            elif assigned_label == 'inc':
                mutated = inject_inconsistency_strict(orig)
            elif assigned_label == 'rep':
                mutated = inject_repetition_contextual(orig)
            elif assigned_label == 'omit':
                mutated = inject_omission(orig, ids)
            counts[assigned_label]+=1
        except Exception as e:
            logs.append({'filename':fname,'error':str(e)})
            counts['clean']+=1

        rows.append({
            'filename': fname,
            'hallucination_type': assigned_label,
            'ast_nodes': ast_nodes,
            'original_code': orig,
            'hallucinated_code': mutated
        })

    df_out=pd.DataFrame(rows, columns=['filename','hallucination_type','ast_nodes','original_code','hallucinated_code'])
    df_out.to_csv(OUTPUT_CSV,index=False)
    pd.DataFrame(logs).to_csv(LOG_CSV,index=False)
    print("Finished. Counts:",counts)
    print(f"Saved -> {OUTPUT_CSV}")

if __name__=="__main__":
    main()


/usr/local/lib/python3.12/dist-packages/tree_sitter/__init__.py:36: FutureWarning: Language(path, name) is deprecated. Use Language(ptr, name) instead.
  warn("{} is deprecated. Use {} instead.".format(old, new), FutureWarning)


Finished. Counts: {'dead': 94, 'inc': 94, 'rep': 94, 'omit': 94, 'clean': 91}
Saved -> /content/final_context_inferred_hallucinated_dataset.csv


In [ ]:
#!/usr/bin/env python3
import os

# ==================== OFFLINE ENV ====================
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_SSL_VERIFY"] = "1"
os.environ["CURL_CA_BUNDLE"] = ""
os.environ["REQUESTS_CA_BUNDLE"] = ""
os.environ["HF_HOME"] = "/root/.cache/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/root/.cache/huggingface"
os.environ["BITSANDBYTES_NOWELCOME"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

# ==================== Paths ====================
DATA_CSV = "/workspace/finetune_training_dataset.csv"
MODEL_NAME_OR_PATH = "/workspace/models/codellama-7b-code"  # 🔹 Offline path inside container
CACHE_DIR = "/root/.cache/huggingface"
OUTPUT_DIR = "/workspace/codellama7b_finetuned_peft"

# ==================== Load dataset ====================
print("Loading dataset...")
df = pd.read_csv(DATA_CSV)
if not {"input_text","target_text"}.issubset(df.columns):
    raise ValueError("Dataset must contain 'input_text' and 'target_text' columns")

dataset = Dataset.from_pandas(df[["input_text","target_text"]])
dataset = dataset.train_test_split(test_size=0.05, seed=42)
train = dataset["train"]
evald = dataset["test"]

# ==================== Load tokenizer ====================
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME_OR_PATH,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# ==================== Prompt function ====================
def make_prompt(ex):
    return {
        "text": (
            "### FILENAME\n"
            f"{ex['input_text'].splitlines()[0]}\n\n"
            "### HALLUCINATED_CPP\n"
            + "\n".join(ex['input_text'].splitlines()[1:]) + "\n\n"
            "### JAVA CODE\n"
            f"{ex['target_text'].strip()}\n"
        )
    }

train = train.map(lambda ex: make_prompt(ex), remove_columns=train.column_names)
evald = evald.map(lambda ex: make_prompt(ex), remove_columns=evald.column_names)

MAX_LENGTH = 4096

# ==================== Tokenization ====================
def tokenize_fn(examples):
    input_ids_batch, labels_batch, attention_masks = [], [], []

    for text in examples["text"]:
        enc = tokenizer(text, truncation=True, max_length=MAX_LENGTH)
        input_ids = enc["input_ids"]

        marker = "### JAVA CODE"
        marker_ids = tokenizer(marker, add_special_tokens=False)["input_ids"]
        boundary_idx = 0
        for i in range(len(input_ids) - len(marker_ids) + 1):
            if input_ids[i:i+len(marker_ids)] == marker_ids:
                boundary_idx = i + len(marker_ids)
                break

        labels = [-100] * len(input_ids)
        for j in range(boundary_idx, len(input_ids)):
            labels[j] = input_ids[j]

        if len(input_ids) < MAX_LENGTH:
            pad_len = MAX_LENGTH - len(input_ids)
            input_ids += [tokenizer.pad_token_id] * pad_len
            labels += [-100] * pad_len
        else:
            input_ids = input_ids[:MAX_LENGTH]
            labels = labels[:MAX_LENGTH]

        attention_mask = [1 if id != tokenizer.pad_token_id else 0 for id in input_ids]

        input_ids_batch.append(input_ids)
        labels_batch.append(labels)
        attention_masks.append(attention_mask)

    return {
        "input_ids": input_ids_batch,
        "labels": labels_batch,
        "attention_mask": attention_masks
    }

train_tokenized = train.map(tokenize_fn, batched=True, remove_columns=["text"])
eval_tokenized = evald.map(tokenize_fn, batched=True, remove_columns=["text"])

# ==================== Load model (4-bit) ====================
print("Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_OR_PATH,
    load_in_4bit=True,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
    cache_dir=CACHE_DIR,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

# ==================== LoRA ====================
candidate_targets = [
    "q_proj","k_proj","v_proj","o_proj",
    "up_proj","down_proj","gate_proj","fc_in","fc_out"
]

found_targets = []
for name, _ in model.named_modules():
    for cand in candidate_targets:
        if cand in name and cand not in found_targets:
            found_targets.append(cand)

if not found_targets:
    found_targets = ["q_proj","k_proj","v_proj","o_proj"]

print("Using LoRA target modules:", found_targets)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=found_targets,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ==================== Training ====================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",
    evaluation_strategy="steps",
    eval_steps=500,
    save_total_limit=2,
    remove_unused_columns=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    tokenizer=tokenizer,
)

print("Starting training...")
trainer.train()

print("Saving LoRA adapters...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved to", OUTPUT_DIR)

In [ ]:
#!/usr/bin/env python3
import os
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# ==================== OFFLINE ENV ====================
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_SSL_VERIFY"] = "1"
os.environ["CURL_CA_BUNDLE"] = ""
os.environ["REQUESTS_CA_BUNDLE"] = ""
os.environ["HF_HOME"] = "/root/.cache/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/root/.cache/huggingface"
os.environ["BITSANDBYTES_NOWELCOME"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# =======================
# PATHS
# =======================
MODEL_NAME_OR_PATH = "/workspace/models/codellama-7b-code"   # 🔹 OFFLINE MODEL
LORA_PATH = "/workspace/codellama7b_finetuned_peft"          # 🔹 LoRA from finetuning
DATA_CSV = "/workspace/finetune_testing_dataset.csv"
OUTPUT_CSV = "/workspace/predicted_10rows.csv"
CACHE_DIR = "/root/.cache/huggingface"

# =======================
# Load tokenizer
# =======================
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME_OR_PATH,
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# =======================
# Load 4-bit model + LoRA
# =======================
print("Loading base model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_OR_PATH,
    load_in_4bit=True,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
    cache_dir=CACHE_DIR,
)

print("Applying LoRA adapters...")
model = PeftModel.from_pretrained(model, LORA_PATH)
model.eval()

# =======================
# Load 10 rows
# =======================
df = pd.read_csv(DATA_CSV)
df_sample = df.sample(10, random_state=42)

# =======================
# Build inference prompt
# =======================
def build_prompt(input_text):
    lines = input_text.splitlines()
    filename = lines[0]
    cpp_code = "\n".join(lines[1:])

    return (
        "### FILENAME\n"
        f"{filename}\n\n"
        "### HALLUCINATED_CPP\n"
        f"{cpp_code}\n\n"
        "### JAVA CODE\n"
        "// Output ONLY valid Java code.\n"
        "// Do not write Python.\n"
    )

# =======================
# Extract ONLY Java code
# =======================
def extract_java(full_output):
    if "### JAVA CODE" not in full_output:
        return ""

    java = full_output.split("### JAVA CODE", 1)[1]

    stop_tokens = [
        "### PYTHON", "### PY", "### CPP", "### C++",
        "import math", "def ", "import numpy", "#!/usr"
    ]

    for token in stop_tokens:
        if token in java:
            java = java.split(token)[0]

    return java.strip()

# =======================
# Inference
# =======================
preds = []

print("\n========== RUNNING ON 10 ROWS ==========\n")

for idx, row in df_sample.iterrows():
    prompt = build_prompt(row["input_text"])

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=1500,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    java_code = extract_java(decoded)

    preds.append({
        "input_text": row["input_text"],
        "predicted_java": java_code,
    })

    print("------------ SAMPLE ------------")
    print("PREDICTED JAVA:\n", java_code)
    print("--------------------------------\n")

# =======================
# Save CSV
# =======================
pd.DataFrame(preds).to_csv(OUTPUT_CSV, index=False)
print("Saved:", OUTPUT_CSV)

In [ ]:
#!/usr/bin/env python3
import os

# ==================== OFFLINE ENV ====================
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_SSL_VERIFY"] = "1"
os.environ["CURL_CA_BUNDLE"] = ""
os.environ["REQUESTS_CA_BUNDLE"] = ""
os.environ["HF_HOME"] = "/root/.cache/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/root/.cache/huggingface"
os.environ["BITSANDBYTES_NOWELCOME"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

# ==================== Paths ====================
DATA_CSV = "/workspace/train_dataset_java.csv"
MODEL_NAME_OR_PATH = "/workspace/models/codellama-7b-code"   # 🔹 OFFLINE MODEL
CACHE_DIR = "/root/.cache/huggingface"
OUTPUT_DIR = "/workspace/codellama7b_finetuned_peft_java"

# ==================== Load dataset ====================
print("Loading dataset...")
df = pd.read_csv(DATA_CSV)

required_cols = {"input_text", "hallucination_label", "target_text"}
if not required_cols.issubset(df.columns):
    raise ValueError("Dataset must contain input_text, hallucination_label, target_text")

dataset = Dataset.from_pandas(df[["input_text", "hallucination_label", "target_text"]])
dataset = dataset.train_test_split(test_size=0.05, seed=42)
train = dataset["train"]
evald = dataset["test"]

# ==================== Load tokenizer ====================
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME_OR_PATH,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# ==================== Prompt function ====================
def make_prompt(ex):
    return {
        "text": (
            "### HALLUCINATION_LABEL\n"
            f"{ex['hallucination_label']}\n\n"
            "### HALLUCINATED_JAVA\n"
            f"{ex['input_text'].strip()}\n\n"
            "### CORRECT_JAVA\n"
            f"{ex['target_text'].strip()}\n"
        )
    }

train = train.map(lambda ex: make_prompt(ex), remove_columns=train.column_names)
evald = evald.map(lambda ex: make_prompt(ex), remove_columns=evald.column_names)

MAX_LENGTH = 4096

# ==================== Tokenization with masking ====================
def tokenize_fn(examples):
    input_ids_batch, labels_batch, attention_masks = [], [], []

    marker = "### CORRECT_JAVA"
    marker_ids = tokenizer(marker, add_special_tokens=False)["input_ids"]

    for text in examples["text"]:
        enc = tokenizer(text, truncation=True, max_length=MAX_LENGTH)
        input_ids = enc["input_ids"]

        boundary_idx = 0
        for i in range(len(input_ids) - len(marker_ids) + 1):
            if input_ids[i:i+len(marker_ids)] == marker_ids:
                boundary_idx = i + len(marker_ids)
                break

        labels = [-100] * len(input_ids)
        for j in range(boundary_idx, len(input_ids)):
            labels[j] = input_ids[j]

        if len(input_ids) < MAX_LENGTH:
            pad_len = MAX_LENGTH - len(input_ids)
            input_ids += [tokenizer.pad_token_id] * pad_len
            labels += [-100] * pad_len
        else:
            input_ids = input_ids[:MAX_LENGTH]
            labels = labels[:MAX_LENGTH]

        attention_mask = [1 if id != tokenizer.pad_token_id else 0 for id in input_ids]

        input_ids_batch.append(input_ids)
        labels_batch.append(labels)
        attention_masks.append(attention_mask)

    return {
        "input_ids": input_ids_batch,
        "labels": labels_batch,
        "attention_mask": attention_masks
    }

train_tokenized = train.map(tokenize_fn, batched=True, remove_columns=["text"])
eval_tokenized = evald.map(tokenize_fn, batched=True, remove_columns=["text"])

# ==================== Load model in 4-bit ====================
print("Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_OR_PATH,
    load_in_4bit=True,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
    cache_dir=CACHE_DIR,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

# ==================== LoRA ====================
candidate_targets = [
    "q_proj","k_proj","v_proj","o_proj",
    "up_proj","down_proj","gate_proj","fc_in","fc_out"
]

found_targets = []
for name, _ in model.named_modules():
    for cand in candidate_targets:
        if cand in name and cand not in found_targets:
            found_targets.append(cand)

if not found_targets:
    found_targets = ["q_proj","k_proj","v_proj","o_proj"]

print("Using LoRA target modules:", found_targets)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=found_targets,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ==================== Training ====================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",
    evaluation_strategy="steps",
    eval_steps=500,
    save_total_limit=2,
    remove_unused_columns=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    tokenizer=tokenizer,
)

print("Starting training...")
trainer.train()

print("Saving LoRA adapters...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved to", OUTPUT_DIR)

In [ ]:
#!/usr/bin/env python3
import os

# ==================== OFFLINE ENV ====================
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_SSL_VERIFY"] = "1"
os.environ["CURL_CA_BUNDLE"] = ""
os.environ["REQUESTS_CA_BUNDLE"] = ""
os.environ["HF_HOME"] = "/root/.cache/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/root/.cache/huggingface"
os.environ["BITSANDBYTES_NOWELCOME"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# ------------------ PATHS ------------------
MODEL_NAME_OR_PATH = "/workspace/models/codellama-7b-code"   # 🔹 OFFLINE MODEL
LORA_PATH = "/workspace/starcoder2_finetuned_peft_java"
DATA_CSV = "/workspace/test_dataset_java.csv"
OUTPUT_CSV = "/workspace/predicted_5rows_new.csv"
CACHE_DIR = "/root/.cache/huggingface"

# ------------------ LOAD TOKENIZER ------------------
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME_OR_PATH,
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ------------------ LOAD 4-BIT MODEL ------------------
print("Loading 4-bit model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_OR_PATH,
    load_in_4bit=True,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
    cache_dir=CACHE_DIR,
)

# ------------------ APPLY LORA ------------------
print("Applying LoRA...")
model = PeftModel.from_pretrained(model, LORA_PATH)
model.eval()

# ------------------ LOAD DATA ------------------
df = pd.read_csv(DATA_CSV)
df_sample = df.sample(5, random_state=42)

# =====================================================
#                   BUILD PROMPT
# =====================================================
def build_prompt(input_text, hallucination_label):
    hallucinated_java = input_text.strip()

    return (
        "### HALLUCINATION_LABEL\n"
        f"{hallucination_label}\n\n"
        "### HALLUCINATED_JAVA\n"
        f"{hallucinated_java}\n\n"
        "### CORRECT_JAVA\n"
        "Provide only corrected Java code. No explanations. No comments.\n"
        "Start directly with import or class.\n\n"
    )

# =====================================================
#               PERFECT JAVA EXTRACTION
# =====================================================
def extract_java(full_output):
    if "### CORRECT_JAVA" not in full_output:
        return ""

    java = full_output.split("### CORRECT_JAVA", 1)[1]

    stop_tokens = [
        "### END",
        "/code/",
        "### HALLUCINATED_JAVA",
        "### FILENAME",
        "```",
        "import sys",
        "def ",
        "### PYTHON",
        "### CPP",
        "### C++"
    ]

    for token in stop_tokens:
        if token in java:
            java = java.split(token)[0]

    bad_prefixes = [
        "Provide only corrected Java code",
        "Start directly",
        "No explanations",
        "No comments"
    ]

    lines = java.splitlines()
    while lines and any(bp in lines[0] for bp in bad_prefixes):
        lines.pop(0)
    java = "\n".join(lines)

    header_patterns = [
        "// Copyright",
        "// All rights reserved",
        "// This source code is licensed",
    ]

    lines = java.splitlines()
    while lines and any(lines[0].strip().startswith(h) for h in header_patterns):
        lines.pop(0)
    java = "\n".join(lines)

    return java.strip()

# =====================================================
#               RUN INFERENCE
# =====================================================
preds = []
print("\n========== RUNNING ON 5 ROWS ==========\n")

for idx, row in df_sample.iterrows():
    prompt = build_prompt(row["input_text"], row["hallucination_label"])
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=1600,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    java_code = extract_java(decoded)

    preds.append({
        "input_text": row["input_text"],
        "predicted_java": java_code,
    })

    print("------------ SAMPLE ------------")
    print(java_code)
    print("--------------------------------\n")

# ------------------ SAVE ------------------
pd.DataFrame(preds).to_csv(OUTPUT_CSV, index=False)
print("Saved:", OUTPUT_CSV)

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import joblib

# ===============================
# 1️⃣ Load training dataset
# ===============================

train_df = pd.read_csv("/workspace/cpp_metrics_train_80.csv")

# ===============================
# 2️⃣ Feature columns (FIXED)
# ===============================

feature_cols = [
    "LOC", "CLOC", "Cyclomatic_Complexity",
    "Function_Count", "Include_Count", "Macro_Count",
    "Pointer_Count", "STL_Usage", "AST_Node_Count",
    "Halstead_Vocabulary", "Halstead_Length",
    "Halstead_Volume", "Halstead_Effort"
]

X_train = train_df[feature_cols]
y_train = train_df["hallucination_label"]

# Convert to numeric & clean (Docker safe)
X_train = X_train.apply(pd.to_numeric, errors="coerce").fillna(0)

# ===============================
# 3️⃣ Standardize features
# ===============================

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# ===============================
# 4️⃣ Train SVM
# ===============================

svm_clf = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    probability=True,
    random_state=42
)

svm_clf.fit(X_train_scaled, y_train)

print("✅ CPP SVM model trained successfully")

# ===============================
# 5️⃣ Save model & scaler
# ===============================

joblib.dump(svm_clf, "/workspace/svm_cpp_hallucination_model.pkl")
joblib.dump(scaler, "/workspace/cpp_scaler.pkl")

print("✅ Model and scaler saved inside /workspace")

In [ ]:
import pandas as pd
import joblib

# ===============================
# 1️⃣ Load test dataset
# ===============================

test_df = pd.read_csv("/workspace/cpp_metrics_test_20.csv")

# ===============================
# 2️⃣ Feature columns (MUST MATCH TRAIN)
# ===============================

feature_cols = [
    "LOC", "CLOC", "Cyclomatic_Complexity",
    "Function_Count", "Include_Count", "Macro_Count",
    "Pointer_Count", "STL_Usage", "AST_Node_Count",
    "Halstead_Vocabulary", "Halstead_Length",
    "Halstead_Volume", "Halstead_Effort"
]

X_test = test_df[feature_cols]

# Convert to numeric & clean
X_test = X_test.apply(pd.to_numeric, errors="coerce").fillna(0)

# ===============================
# 3️⃣ Load trained model & scaler
# ===============================

svm_clf = joblib.load("/workspace/svm_cpp_hallucination_model.pkl")
scaler = joblib.load("/workspace/cpp_scaler.pkl")

# ===============================
# 4️⃣ Scale & predict
# ===============================

X_test_scaled = scaler.transform(X_test)

preds = svm_clf.predict(X_test_scaled)
probs = svm_clf.predict_proba(X_test_scaled)

# Confidence of predicted class
pred_conf = probs.max(axis=1)

# ===============================
# 5️⃣ Save ONLY required columns
# ===============================

output_df = pd.DataFrame({
    "code": test_df["cppcode"],   # <-- original C++ code column
    "predicted_label": preds,
    "predicted_confidence": pred_conf
})

output_df.to_csv("/workspace/cpp_metrics_test_20_predicted.csv", index=False)

print("\n===================================")
print("✅ Predictions saved to /workspace/cpp_metrics_test_20_predicted.csv")
print("===================================\n")

print(output_df.head())

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import joblib

# ===============================
# 1️⃣ Load full training dataset
# ===============================

df = pd.read_csv("java_metrics_train_output.csv")

# ===============================
# 2️⃣ Prepare features & labels
# ===============================

feature_cols = [
    "Class", "LOC", "CLOC", "Cyclomatic Complexity",
    "Number of Methods", "WMC", "DIT", "NOC", "CBO",
    "RFC", "LCOM", "Halstead Vocabulary", "Halstead Length",
    "Halstead Volume", "Halstead Effort"
]

X = df[feature_cols]
y = df["hallucination_label"]

# Convert to numeric and fill NaNs
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)

# ===============================
# 3️⃣ Standardize features
# ===============================

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ===============================
# 4️⃣ Train SVM model
# ===============================

clf = SVC(kernel="rbf", C=10, gamma=0.1, probability=True, random_state=42)
clf.fit(X_scaled, y)

# ===============================
# 5️⃣ Save model & scaler
# ===============================

joblib.dump(clf, "/workspace/svm_hallucination_model.pkl")
joblib.dump(scaler, "/workspace/scaler.pkl")

print("✅ SVM model and scaler saved inside /workspace")

In [ ]:
import pandas as pd
import joblib

# ===============================
# 1️⃣ Load test dataset & model
# ===============================

test_df = pd.read_csv("/workspace/java_metrics_test_output.csv")

model = joblib.load("/workspace/svm_hallucination_model.pkl")
scaler = joblib.load("/workspace/scaler.pkl")

# ===============================
# 2️⃣ Prepare features
# ===============================

# Feature columns used in training
feature_cols = [
    "Class", "LOC", "CLOC", "Cyclomatic Complexity",
    "Number of Methods", "WMC", "DIT", "NOC", "CBO",
    "RFC", "LCOM", "Halstead Vocabulary", "Halstead Length",
    "Halstead Volume", "Halstead Effort"
]

X_test = test_df[feature_cols]

# Convert to numeric and fill NaNs
X_test = X_test.apply(pd.to_numeric, errors="coerce").fillna(0)

# Scale features
X_test_scaled = scaler.transform(X_test)

# ===============================
# 3️⃣ Predict hallucination labels
# ===============================

preds = model.predict(X_test_scaled)
probs = model.predict_proba(X_test_scaled).max(axis=1)

# ===============================
# 4️⃣ Save predictions
# ===============================

test_df["predicted_label"] = preds
test_df["prediction_confidence"] = probs

# Keep only javacode + predictions in output
output_df = test_df[["javacode", "predicted_label", "prediction_confidence"]]

output_df.to_csv("/workspace/test_predictions_svm.csv", index=False)

print("\n✅ Predictions saved to /workspace/test_predictions_svm.csv\n")
print(output_df.head())

In [ ]:
import pandas as pd
import re
from difflib import SequenceMatcher
from tree_sitter import Language, Parser
import tree_sitter_java
from codebleu import calc_codebleu

# ==============================
# LOAD DATA
# ==============================
df = pd.read_csv("/content/merged_dataset_codegen_cj.csv")

# ==============================
# TREE-SITTER SETUP
# ==============================
JAVA_LANGUAGE = Language(tree_sitter_java.language())
parser = Parser()
parser.set_language(JAVA_LANGUAGE)

# ==============================
# CLEAN CODE
# ==============================
def clean_code(code):
    code = str(code)

    # Remove markdown blocks
    code = re.sub(r"```.*?```", "", code, flags=re.DOTALL)
    code = code.replace("```java", "")
    code = code.replace("```", "")

    return code.strip()

# ==============================
# AST TOKEN EXTRACTION
# ==============================
def get_ast_tokens(code):
    try:
        code = clean_code(code)
        tree = parser.parse(bytes(code, "utf8"))
        tokens = []

        def traverse(node):
            tokens.append(node.type)
            for child in node.children:
                traverse(child)

        traverse(tree.root_node)
        return tokens
    except:
        return []

def compute_ast_similarity(code1, code2):
    tokens1 = get_ast_tokens(code1)
    tokens2 = get_ast_tokens(code2)

    if not tokens1 or not tokens2:
        return 0.0

    return SequenceMatcher(None, tokens1, tokens2).ratio()

# ==============================
# METRIC COMPUTATION
# ==============================
results = []

for idx, row in df.iterrows():

    original_code = clean_code(row["original_java"])
    predicted_code = clean_code(row["predicted_java"])

    # AST Similarity
    ast_sim = compute_ast_similarity(original_code, predicted_code)

    # CodeBLEU (ignore dataflow)
    try:
        codebleu_score = calc_codebleu(
            [[original_code]],
            [predicted_code],
            lang="java",
            weights=(0.33, 0.33, 0.34, 0.0)   # ignore dataflow
        )["codebleu"]
    except Exception as e:
        print("CodeBLEU error:", e)
        codebleu_score = 0.0

    results.append({
        "class_name": row["class_name"],
        "ast_similarity": ast_sim,
        "codebleu": codebleu_score
    })

    if idx % 20 == 0:
        print(f"Processed {idx}")

# ==============================
# SAVE OUTPUT
# ==============================
df_results = pd.DataFrame(results)
df_results.to_csv("results_codeLlama_cj.csv", index=False)

print("✅ Metrics saved to results_codeLlama_cj.csv ")

/tmp/ipython-input-169/1990475388.py:18: DeprecationWarning: Parser.set_language() is deprecated. Use the language setter instead.
  parser.set_language(JAVA_LANGUAGE)


Processed 0
✅ Metrics saved to results_codeLlama_cj.csv 
